# Шаг 1: zero-shot бейзлайн

PaddleOCR angle-classifier `ch_ppocr_mobile_v2.0_cls` (MobileNetV3, 0.57 MB ONNX) + антисимметричный TTA

In [ ]:
REPO_URL = 'https://github.com/val3rkq/avito.ds.bootcamp.cv.git'
!git clone -q $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
import os, glob, zipfile, subprocess
from google.colab import drive

drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/ml/avito.ds/test.zip'
assert os.path.exists(ZIP), ZIP

with zipfile.ZipFile(ZIP) as z: 
    z.extractall('/content/data')

print(subprocess.run(['find', '/content/data', '-maxdepth', '3', '-not', '-name', '*.png'], capture_output=True, text=True).stdout)

In [ ]:
pngs = glob.glob('/content/data/**/*.png', recursive=True)
IMAGES = os.path.dirname(pngs[0])
print(len(pngs), 'images in', IMAGES)
ss = glob.glob('/content/data/**/sample_submission.csv', recursive=True)
SAMPLE = ss[0] if ss else None
print('sample_submission:', SAMPLE)

In [ ]:
!python -m src.baseline_paddle --images "$IMAGES" {('--sample-submission ' + SAMPLE) if SAMPLE else ''} \n    --out submission.csv --raw-out outputs_raw_v1.csv

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

sub = pd.read_csv('submission.csv')
print(sub.shape, sub.columns.tolist())
print(sub.head())

assert sub.shape == (20000, 2) and sub.p_180.between(0, 1).all() and sub.p_180.notna().all()
sub.p_180.hist(bins=50)
plt.title('p_180 on test')
plt.show()

In [ ]:
from google.colab import files
files.download('submission.csv')